# PR0302: Lectura avanzada de datos de archivos

In [1]:
import pandas as pd

## Lectura CSV

In [26]:
df_norte = pd.read_csv("ventas_norte_v2.csv", sep=";")

df_norte['total_venta'] = (df_norte['Total_Factura']
                           .str.replace('$', '', regex=False)
                           .str.replace(',', '', regex=False)
                           .astype(float))

df_norte = df_norte.rename(columns={
    'ID_Pedido': 'id_transaccion',
    'Fecha_Transaccion': 'fecha',
    'Producto': 'producto',
    'Unidades': 'cantidad'
})
df_norte['region'] = 'Norte'
df_norte['ciudad'] = ''
df_norte.head()

,id_transaccion,fecha,Cliente_Nombre,Direccion_Envio,producto,cantidad,Precio_Unitario,Total_Factura,total_venta,region,ciudad
0,N-1000,2023-05-10 03:00:00,Usuario_0,"Gran Vía, 49, Piso 4",Laptop Gamer,1,970,$970.00,970.0,Norte,
1,N-1001,2023-06-18 09:00:00,Usuario_1,"Av. Libertad, 30, Piso 4",Mouse Ergonómico,4,927,$3708.00,3708.0,Norte,
2,N-1002,2023-03-10 05:00:00,Usuario_2,"Av. Libertad, 98, Piso 2",Monitor 4K,2,1410,$2820.00,2820.0,Norte,
3,N-1003,2023-05-10 03:00:00,Usuario_3,"Paseo Gracia, 94, Piso 7",Webcam HD,2,1269,$2538.00,2538.0,Norte,
4,N-1004,2023-02-25 17:00:00,Usuario_4,"Calle Mayor, 80, Piso 8",Webcam HD,2,454,$908.00,908.0,Norte,


## Lectura Excel

In [24]:
df_sur_dict = pd.read_excel("ventas_sur_v2.xlsx", sheet_name=None)
df_sur = pd.concat(df_sur_dict.values(), ignore_index=True)

df_sur['total_venta'] = (df_sur['Precio_Base'] * df_sur['Cantidad'] * (1 - df_sur['Descuento_Aplicado']))

df_sur = df_sur.rename(columns={
    'Ref_Venta': 'id_transaccion',
    'Fecha_Alta': 'fecha',
    'Articulo': 'producto',
    'Cantidad': 'cantidad',
})
df_sur['region'] = 'Sur'
df_sur['ciudad'] = ''
df_sur.head()

,id_transaccion,fecha,producto,cantidad,Precio_Base,Descuento_Aplicado,Es_Cliente_Corporativo,Estado_Envio,total_venta,region,ciudad
0,S-5000,2023-02-11 00:00:00,Webcam HD,9,425.12,0.20,False,Enviado,3060.864,Sur,
1,S-5001,2023-02-03 12:00:00,Teclado Mecánico,8,599.60,0.00,True,Devuelto,4796.800,Sur,
2,S-5002,2023-04-06 07:00:00,Mouse Ergonómico,6,971.36,0.00,False,Completado,5828.160,Sur,
3,S-5003,2023-04-02 05:00:00,Monitor 4K,1,799.66,0.10,True,Pendiente,719.694,Sur,
4,S-5004,2023-06-17 12:00:00,Mouse Ergonómico,2,619.99,0.05,True,Devuelto,1177.981,Sur,


## Lectura JSON

In [22]:
import pandas as pd
import json

with open('ventas_este_v2.json', 'r') as f:
    data_json = json.load(f)

df_este = pd.json_normalize(data_json, sep='_')

df_este.columns = df_este.columns.str.replace('data_payload_transaccion_detalles_producto_', '')
df_este.columns = df_este.columns.str.replace('data_payload_transaccion_', '')
df_este.columns = df_este.columns.str.replace('data_payload_', '')
df_este.columns = df_este.columns.str.replace('comprador_ubicacion_', '')

df_este['total_venta'] = df_este['precio_lista'] + df_este['impuestos_monto_iva']


df_este = df_este.rename(columns={
    'sku': 'id_transaccion',
    'fecha_evento': 'fecha',
    'nombre_comercial': 'producto',
    'cantidad_comprada': 'cantidad',
    'ciudad': 'ciudad'
})

df_este['region'] = 'Este'

columnas_finales = ['id_transaccion', 'fecha', 'region', 'producto', 'cantidad', 'total_venta', 'ciudad']
df_este = df_este[columnas_finales]

df_este.head()

,id_transaccion,fecha,region,producto,cantidad,total_venta,ciudad
0,G6YATHNF,2023-06-29 00:00:00,Este,Monitor 4K,1,608.63,Sevilla
1,THC8MFC5,2023-04-12 00:00:00,Este,Webcam HD,1,1357.62,Sevilla
2,6IFCBHMJ,2023-04-20 00:00:00,Este,Webcam HD,2,1202.74,Sevilla
3,PXC40RUN,2023-04-14 00:00:00,Este,Webcam HD,2,1518.55,Madrid
4,3JI7HLD5,2023-01-03 00:00:00,Este,Monitor 4K,2,1764.18,Sevilla


In [25]:
columnas_target = [
    'id_transaccion', 'fecha', 'region', 'producto', 
    'cantidad', 'total_venta', 'ciudad'
]

df_total = pd.concat([
    df_norte[columnas_target],
    df_sur[columnas_target],
    df_este[columnas_target]
], ignore_index=True)

df_total['fecha'] = pd.to_datetime(df_total['fecha'], errors='coerce')

df_total.to_csv("ventas_consolidadas_v2.csv", sep=",", encoding="utf-8", index=False)

print("Proceso PR0302 finalizado con éxito.")
print(df_total.head())

Proceso PR0302 finalizado con éxito.
  id_transaccion               fecha region          producto  cantidad  \
0         N-1000 2023-05-10 03:00:00  Norte      Laptop Gamer         1   
1         N-1001 2023-06-18 09:00:00  Norte  Mouse Ergonómico         4   
2         N-1002 2023-03-10 05:00:00  Norte        Monitor 4K         2   
3         N-1003 2023-05-10 03:00:00  Norte         Webcam HD         2   
4         N-1004 2023-02-25 17:00:00  Norte         Webcam HD         2   

   total_venta ciudad  
0        970.0         
1       3708.0         
2       2820.0         
3       2538.0         
4        908.0         
